In [ ]:
# Lesson 6: Essay Writer

### 本节课的核心思路：多节点协作的"写作 Agent"（Plan-Execute-Reflect 模式）

前几课的 Agent 都是"LLM 节点 <-> 工具节点"的简单循环，这一课把 LangGraph 用来搭一个更复杂的多步骤工作流，
模拟一个人写文章的完整流程：

1. **planner**：先根据主题写一个大纲（不查资料，纯规划）
2. **research_plan**：根据主题生成搜索关键词，调用 Tavily 查资料，作为写作素材
3. **generate**：结合大纲 + 素材，写出一版文章草稿
4. **reflect**：像老师批改作文一样，给草稿写点评（critique）
5. **research_critique**：根据点评里提到的问题，再去查一些补充资料
6. 带着新素材回到 **generate** 重新生成草稿，如此循环，直到修改次数（`revision_number`）超过上限（`max_revisions`）才结束

这体现了 LangGraph 的一个重要能力：状态图不一定是简单的"问答循环"，也可以编排成任意复杂的、
带循环、带分支的多步骤工作流，每个节点各司其职，通过共享的 `AgentState` 传递数据。

In [ ]:
from dotenv import load_dotenv
_ = load_dotenv()  # 可选：本方案用本地 Ollama + DuckDuckGo，不再需要 OPENAI_API_KEY / TAVILY_API_KEY

# ==== 本地无密钥模型配置（Ollama + Qwen）====
# 前提：安装 Ollama（https://ollama.com）并执行 `ollama pull qwen2.5`
OLLAMA_BASE_URL = "http://localhost:11434/v1"  # Ollama 的 OpenAI 兼容端点
OLLAMA_API_KEY = "ollama"                        # 占位符，Ollama 不校验密钥
MODEL = "qwen2.5"                                # 中文模型

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

# 【版本变化修复】同 Lesson 4/5：新版 SqliteSaver.from_conn_string(":memory:") 返回的是上下文管理器，
# 不是可以直接用的 SqliteSaver 实例，需要 .__enter__() 拿到真正的实例，
# 并且要保留上下文管理器对象（_memory_cm）避免被垃圾回收提前关闭底层连接，
# 否则后面 graph.stream(...) 真正读写检查点时会报 AttributeError 或 "Cannot operate on a closed database."
_memory_cm = SqliteSaver.from_conn_string(":memory:")
memory = _memory_cm.__enter__()

In [ ]:
class AgentState(TypedDict):
    task: str              # 用户提出的写作任务/主题
    plan: str               # planner 节点产出的大纲
    draft: str               # generate 节点产出的文章草稿
    critique: str             # reflect 节点产出的批改意见
    content: List[str]         # 搜索到的参考资料片段（research_plan / research_critique 不断往里追加）
    revision_number: int        # 当前是第几次修改稿
    max_revisions: int           # 最多允许修改几次，超过就结束（这几个字段都没有自定义 reducer，节点返回啥就直接覆盖）

In [ ]:
from langchain_openai import ChatOpenAI
# 指向本地 Ollama（无需密钥）；temperature=0：写作规划/批改这类任务也希望结果尽量稳定
model = ChatOpenAI(model=MODEL, base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY, temperature=0)

In [ ]:
PLAN_PROMPT = """你是一位资深写作专家，任务是为一篇文章撰写高层次的大纲。\
请针对用户提供的主题写出这样一份大纲。给出文章的大纲，并为各个部分附上相关的备注或写作说明。请用中文。"""

In [ ]:
WRITER_PROMPT = """你是一位写作助手，任务是写出优秀的五段式文章。\
请针对用户的需求和最初的大纲，写出尽可能好的文章。\
如果用户提供了批改意见，请据此给出修改后的新版本。\
按需充分利用下面提供的参考资料，并用中文写作：

------

{content}"""

In [ ]:
REFLECTION_PROMPT = """你是一位批改作文的老师。\
请针对用户提交的文章给出点评和改进建议。\
提供详细的建议，包括对篇幅、深度、文风等方面的要求。请用中文。"""

In [ ]:
RESEARCH_PLAN_PROMPT = """你是一名研究员，负责为撰写下面这篇文章提供可用的信息。\
请生成一组搜索关键词，用来收集任何相关信息。最多只生成 3 条关键词。"""

In [ ]:
RESEARCH_CRITIQUE_PROMPT = """你是一名研究员，负责为完成下面提出的修改要求提供可用的信息。\
请生成一组搜索关键词，用来收集任何相关信息。最多只生成 3 条关键词。"""

In [ ]:
# 【版本变化修复】原代码写的是 `from langchain_core.pydantic_v1 import BaseModel`。
# 这是旧版 langchain 为了兼容 pydantic v1 用户提供的一个内部兼容层，
# 当前安装的 langchain-core（配合 langchain 1.x）已经不再提供这个模块，
# 直接 import 会报 ModuleNotFoundError: No module named 'langchain_core.pydantic_v1'。
# 新版 LangChain 全面基于 pydantic v2，直接从 pydantic 官方包导入 BaseModel 即可。
from pydantic import BaseModel

class Queries(BaseModel):
    queries: List[str]   # 用结构化输出（model.with_structured_output）让模型直接返回符合这个 schema 的搜索关键词列表

In [ ]:
# 【改为无密钥搜索】原课程用 Tavily（需 TAVILY_API_KEY）。这里用 DuckDuckGo 封装一个
# 接口兼容的小客户端：提供 .search(query, max_results) 方法，返回 {"results": [{"content": ...}]}，
# 这样下面 research 节点里的调用代码几乎不用改。需先 `pip install duckduckgo-search`（或 ddgs）。
from duckduckgo_search import DDGS

class DuckDuckGoResearchClient:
    def __init__(self):
        self._ddg = DDGS()  # 构造本身不发请求，第一次调用 .text() 时才联网
    def search(self, query, max_results=2):
        try:
            hits = self._ddg.text(query, max_results=max_results) or []
        except Exception as e:
            print(f"DuckDuckGo 搜索出错，返回空结果：{e}")
            hits = []
        # 把 DuckDuckGo 的 {title, href, body} 统一成 Tavily 风格的 {"results": [{"content": ...}]}，
        # 使 research_plan_node / research_critique_node 里的 response['results'] 和 r['content'] 无需改动
        return {"results": [{"content": h.get("body", "")} for h in hits]}

tavily = DuckDuckGoResearchClient()  # 变量名仍叫 tavily，保持后续节点代码不变

In [ ]:
def plan_node(state: AgentState):
    # planner 节点：只用 task（主题）生成大纲，不依赖任何搜索结果
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

In [ ]:
def research_plan_node(state: AgentState):
    # 让模型生成最多 3 条搜索关键词（结构化输出成 Queries 对象，而不是让模型输出自然语言再自己解析）
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []   # 兼容 content 还没初始化（第一次调用时是 None）的情况
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])   # 把每条搜索结果的正文内容都收集起来，作为后面写作的素材
    return {"content": content}

In [ ]:
def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\n这是我的写作大纲：\n\n{state['plan']}")
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)   # 把搜集到的素材塞进 system prompt，让模型写作时有据可依
        ),
        user_message
        ]
    response = model.invoke(messages)
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1   # 每生成一版草稿，修改计数 +1
    }

In [ ]:
def reflection_node(state: AgentState):
    # reflect 节点：像老师批改作文一样，针对当前草稿给出点评和修改建议
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [ ]:
def research_critique_node(state: AgentState):
    # 和 research_plan_node 几乎一样，区别是这次是根据"批改意见"（critique）而不是"原始主题"来生成搜索关键词，
    # 目的是针对性地补充点评里提到的信息缺口
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def should_continue(state):
    # 条件边：修改次数超过上限就结束，否则继续走 reflect（批改）这条路，进入下一轮修改
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

In [ ]:
builder = StateGraph(AgentState)

In [ ]:
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [ ]:
builder.set_entry_point("planner")  # 整个流程从 planner（先列大纲）开始

In [ ]:
builder.add_conditional_edges(
    "generate",
    should_continue,        # 每生成一版草稿后，判断是结束（END）还是继续走 reflect 修改
    {END: END, "reflect": "reflect"}
)


In [ ]:
builder.add_edge("planner", "research_plan")     # 大纲写完 -> 去查资料
builder.add_edge("research_plan", "generate")      # 资料查完 -> 生成草稿

builder.add_edge("reflect", "research_critique")     # 批改完 -> 针对批改意见再查补充资料
builder.add_edge("research_critique", "generate")      # 补充资料查完 -> 回到 generate 重新生成，形成"修改循环"

In [ ]:
graph = builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image
# 【运行环境问题修复】同前几课：draw_png() 依赖本地未安装的 pygraphviz，会直接 ImportError，
# 这里改用不需要额外依赖、纯本地生成文本的 draw_mermaid()
print(graph.get_graph().draw_mermaid())

In [ ]:
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    'task': "langchain 和 langsmith 有什么区别？",
    "max_revisions": 2,     # 最多修改 2 轮
    "revision_number": 1,   # 从第 1 版草稿开始计数
}, thread):
    print(s)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# 【环境限制，无法在本地修复】helper.py 是 DeepLearning.AI 课程平台自带的辅助文件（封装了一个基于 Gradio 的
# 可视化界面：ewriter 把上面搭的 graph 包一层交互式 UI，writer_gui 用来启动这个 UI），
# 只有在 DeepLearning.AI 官方 Jupyter 环境里才会预置这个文件，直接下载/导出的 .ipynb 并不包含它。
# 本地这个 venv 目录下没有 helper.py，所以下面这行会报 ModuleNotFoundError: No module named 'helper'，
# 这不是代码逻辑错误，而是缺少课程平台专属的配套文件，无法通过修改这个 notebook 本身来解决。
# 如果你想在本地跑起来，需要自己实现一个简单的 Gradio 界面来调用上面已经定义好的 graph（graph.stream / graph.get_state 等）。
from helper import ewriter, writer_gui

In [ ]:
MultiAgent = ewriter()          # 用 helper.py 里的 ewriter 把上面的 graph 包装成一个带交互式 UI 的对象
app = writer_gui(MultiAgent.graph)   # 启动 Gradio 界面，可以在网页上输入主题、逐步查看 plan/draft/critique 的迭代过程
app.launch()